#### Module 5 — Implementation and 5G RAN Accessibility Causal Observability Reference V1

**Structured filename:** `M05_00_Implementation_and_Causal_Observability_Reference_v1.ipynb`

This notebook is the **design, implementation-roadmap and causal-observability reference** for Module 5 of the Telecom AI Engineering Platform.

It is intentionally separate from executable notebooks. Its job is to explain:

- what Module 5 is trying to prove;
- how the end-to-end agentic architecture is structured;
- which implementation notebooks will be built and in what order;
- what has already been frozen;
- how 5G RAN Accessibility is decomposed into diagnosable investigation components;
- which standardized, diagnostic and contextual evidence must exist;
- how synthetic V1, executable V2 and future live/collected telemetry remain aligned;
- how incident truth, agent-visible evidence, ML inference and action policy remain separated;
- and what criteria must be satisfied before a fault family enters the formal benchmark.

> **North Star:** The LLM orchestrates. Deterministic telemetry supplies network truth. Standards knowledge provides grounding. ML supplies probabilistic evidence. Policy controls actions. Verification closes the loop.


#### Navigation

- [Section 1 — Module 5 Context and Research Objective](#section-1--module-5-context-and-research-objective)
- [Section 2 — End-to-End Agentic Architecture](#section-2--end-to-end-agentic-architecture)
- [Section 3 — Implementation Roadmap and Notebook Plan](#section-3--implementation-roadmap-and-notebook-plan)
- [Section 4 — Frozen Foundation Snapshot](#section-4--frozen-foundation-snapshot)
- [Section 5 — Synthetic V1 Design Philosophy](#section-5--synthetic-v1-design-philosophy)
- [Section 6 — Accessibility KPI and Investigation Scope](#section-6--accessibility-kpi-and-investigation-scope)
- [Section 7 — Canonical Evidence Catalogue](#section-7--canonical-evidence-catalogue)
- [Section 8 — Component-by-Component Causal Observability](#section-8--component-by-component-causal-observability)
- [Section 9 — Cross-Component Observability Matrix](#section-9--cross-component-observability-matrix)
- [Section 10 — Fault-Family and Incident Benchmark Design](#section-10--fault-family-and-incident-benchmark-design)
- [Section 11 — Healthy Baseline and KPI Validation Plan](#section-11--healthy-baseline-and-kpi-validation-plan)
- [Section 12 — Independent Simulator Reference Validation](#section-12--independent-simulator-reference-validation)
- [Section 13 — Synthetic V1 → Executable V2 → Live Alignment](#section-13--synthetic-v1--executable-v2--live-alignment)
- [Section 14 — MCP and Agent Investigation Plan](#section-14--mcp-and-agent-investigation-plan)
- [Section 15 — ML, Policy, Action and Verification Plan](#section-15--ml-policy-action-and-verification-plan)
- [Section 16 — Evaluation Framework](#section-16--evaluation-framework)
- [Section 17 — Master Readiness Checklist](#section-17--master-readiness-checklist)
- [Section 18 — Provenance, Reproducibility and Safety Rules](#section-18--provenance-reproducibility-and-safety-rules)
- [Section 19 — Current Status and Immediate Next Step](#section-19--current-status-and-immediate-next-step)
- [Appendix A — Reference Configuration Skeleton](#appendix-a--reference-configuration-skeleton)


#### Section 1 — Module 5 Context and Research Objective

##### Project evolution

| Module | Architecture |
|---|---|
| **Module 1** | Standalone LLM |
| **Module 2** | Telecom Semantic RAG |
| **Module 3** | Knowledge-Based MCP |
| **Module 4** | Adaptive RAG + Knowledge-Based MCP Hybrid |
| **Module 5** | **Agentic Telecom Investigation — Knowledge + Telemetry + ML** |
| **Module 6** | Production & Predictive Telecom AI + RAG V2 |
| **Module 7** | Controlled Autonomous Telecom Operations |

Modules 1–4 primarily answered engineering questions from knowledge. Module 5 changes the unit of work from a **question** to a **network incident**.

```text
MODULES 1–4
Question / Engineering Problem
        ↓
Knowledge Retrieval
        ↓
Evidence Synthesis
        ↓
Answer
```

becomes:

```text
MODULE 5
Network Incident
      ↓
Observe Network State
      ↓
Select Investigation Step
      ↓
Knowledge / Telemetry / ML
      ↓
Update Evidence + Hypotheses
      ↓
Evidence Sufficient?
   ┌──────┴──────┐
   │             │
  NO            YES
   │             │
Investigate    Diagnose
Further          ↓
             Action Needed?
              ┌───┴───┐
              │       │
             NO      YES
              │       │
            Report  Policy Gate
                       ↓
                Approved Action
                       ↓
                 Observe Effect
                       ↓
                    Verify
```

##### Primary research question

> **Can an LLM-driven telecom agent autonomously select and sequence standards-grounded knowledge, hierarchical RAN telemetry and specialist ML tools to diagnose realistic 5G RAN incidents accurately, efficiently and safely?**

##### Key secondary questions

1. Can the agent progressively narrow an investigation from KPI → component → underlying evidence?
2. Can it decide when knowledge, telemetry or ML evidence is actually required?
3. Can it avoid unnecessary retrieval, unnecessary deep measurements and premature diagnosis?
4. Can it distinguish shared-domain failures from cell-local symptoms?
5. Can it treat ML as probabilistic evidence rather than authoritative truth?
6. Can it recover from unavailable, misleading or contradictory tools?
7. Can it recommend or execute only actions permitted by deterministic policy?
8. What trade-offs emerge between diagnostic quality, tool efficiency, latency and token consumption?


#### Section 2 — End-to-End Agentic Architecture

##### Core architecture

```text
                    TELECOM AI ENGINEERING PLATFORM

                           LLM AGENT
                      Granite / vLLM
                              │
                              ▼
                          LANGGRAPH
                  state + workflow + branching
                              │
                              ▼
                         MCP CLIENT
                              │
        ┌─────────────────────┼────────────────────────┐
        │                     │                        │
        ▼                     ▼                        ▼
 KNOWLEDGE PLANE       OBSERVATION PLANE        INFERENCE PLANE
        │                     │                        │
 Semantic RAG MCP       Telemetry MCP                 ML MCP
 Knowledge MCP          KPI / PM / alarms             │
 Standards              topology / resources          ├─ anomaly
 Procedures             config / events               └─ fault/RCA
        │                     │                        │
        └─────────────────────┼────────────────────────┘
                              ▼
                     EVIDENCE / AGENT STATE
                              │
                              ▼
                          DIAGNOSIS
                              │
                              ▼
                       RECOMMENDATION
                              │
                              ▼
                     DETERMINISTIC POLICY
                              │
                    ┌─────────┼─────────┐
                    │         │         │
                  ALLOW    APPROVAL    DENY
                    │         │
                    └────┬────┘
                         ▼
                      ACTION MCP
                         │
                         ▼
                   OBSERVE OUTCOME
                         │
                         ▼
                       VERIFY
```

##### Responsibilities by technology

| Layer | Role in Module 5 |
|---|---|
| **Granite / vLLM** | reason over evidence and choose investigation steps |
| **LangGraph** | own state, sequencing, branching, loops and stop conditions |
| **LangChain** | thin model/tool abstraction and MCP interoperability |
| **MCP** | standardized capability boundary for knowledge, telemetry, ML and actions |
| **LangSmith** | agent trajectory tracing and debugging |
| **OpenTelemetry** | system/runtime latency, service and tool observability |
| **Pydantic** | typed state, tool and observation contracts |
| **Deterministic policy** | authorize, require approval or deny actions |
| **Independent experiment harness** | score correctness, efficiency and safety independently of the agent |

##### Separation of responsibilities

```text
LangGraph
"What should happen next?"

MCP
"How is the selected capability invoked?"

Telemetry
"What is the observed network state?"

ML
"What probabilistic pattern is suggested?"

Policy
"Is the proposed action permitted?"

Verification
"Did the network state actually improve?"
```

> **Agentic AI augments the mature engineering process selectively; it does not replace deterministic calculations, telemetry retrieval, policy enforcement or verification with free-form LLM judgement.**


#### Section 3 — Implementation Roadmap and Notebook Plan

The reference notebook is numbered `M05_00` because it is the architectural guide rather than an execution stage.

##### Planned notebook sequence

| Notebook | Primary purpose | Status |
|---|---|---|
| `M05_00_Implementation_and_Causal_Observability_Reference_v1.ipynb` | architecture, roadmap, causal-observability reference | **Current reference** |
| `M05_01_Foundation_and_Healthy_Telemetry_v1.ipynb` | schemas, Accessibility graph, topology, observability config, healthy telemetry, freeze manifest | **Complete / frozen** |
| `M05_02_Runtime_Restore_KPI_Derivation_and_Healthy_Validation_v1.ipynb` | restore root manifest, load baseline, derive formal KPIs, validate healthy behaviour | **Next** |
| `M05_03_Sionna_Radio_Reference_Validation_v1.ipynb` | independent radio/system-level sanity reference against selected synthetic relationships | Planned |
| `M05_04_Controlled_Incident_Generation_and_Benchmark_v1.ipynb` | inject diagnosable faults, create hidden truth, benchmark scenarios and distractors | Planned |
| `M05_05_Telemetry_MCP_Service_v1.ipynb` | expose hierarchical network observations through deterministic MCP tools | Planned |
| `M05_06_Specialist_ML_MCP_v1.ipynb` | anomaly/fault specialist models exposed as probabilistic MCP evidence | Planned |
| `M05_07_LangGraph_Agentic_Investigation_v1.ipynb` | Granite agent state, tool selection, evidence sufficiency, RCA and trajectory logging | Planned |
| `M05_08_Policy_Action_MCP_and_Verification_v1.ipynb` | deterministic action gates, controlled simulated actions and post-action verification | Planned |
| `M05_09_Module5_Evaluation_and_Closeout_v1.ipynb` | formal cross-scenario/model evaluation, efficiency, safety and conclusions | Planned |

##### Artifact handoff

```text
M05_01 FOUNDATION
      │
      ├── canonical graph
      ├── observation contract
      ├── causal topology
      ├── observability config
      ├── healthy telemetry
      └── root manifest
              │
              ▼
M05_02 KPI VALIDATION
              │
              ├──────────────► M05_03 SIONNA REFERENCE
              │
              ▼
M05_04 INCIDENT BENCHMARK
              │
              ▼
M05_05 TELEMETRY MCP
              │
        ┌─────┴─────┐
        ▼           ▼
 Knowledge MCP    M05_06 ML MCP
        │           │
        └─────┬─────┘
              ▼
M05_07 LANGGRAPH AGENT
              │
              ▼
M05_08 POLICY / ACTION / VERIFY
              │
              ▼
M05_09 FORMAL EVALUATION
```

Notebook boundaries are deliberate: **build-time artifacts are frozen once, while later notebooks restore and consume them rather than reconstructing architecture from notebook state.**


#### Section 4 — Frozen Foundation Snapshot

The completed foundation notebook currently establishes the following semantic baseline.

##### Frozen semantic identities

```text
Accessibility Graph V1.2
aaeb1acf5739a9d80d7745469faa71fe142181c7e96df6bc2c9e5f8b8b031b68

KPI Knowledge Supplement V1
a205b4ab672eabe90380fd2bcc7dda81ef4b125c6eb278e69ab0c462d3949348

Observation Contract V1.1
07ba4e0cc0f672cc03268e96bb53aa0f322241be43a4f957e7065b0d6ebae312

Synthetic Causal Topology V1.1
b12fd1cc5b208963a13caecc33d03016a612ff1f1f112686f4db435db66b421d

Causal Observability Config V1
8ccce01ee1375bbef52611a4b008ea0138bae88874993be67f7577aa0838f42a

Healthy Dataset V1
fb516b29c18b68b1dc5ce2fd900bd0b6dc888c1361607c7626e7c326114cdc95

Module 5 Foundation Root
ceab50e83ff910d439fdcd98ce64216e1e86548ec000966dcd1906154e7c3309
```

##### Frozen scale

```text
Formal TS 28.554 RAN Accessibility KPIs : 2
Investigation components                : 7
Standardized metric families            : 21
Operational metric families             : 49
Total observable metric families        : 70
Observation entities                    : 132
Unique fault-family concepts            : 37
Component/fault-family mappings         : 58

Healthy baseline duration               : 7 days
PM interval                             : 15 minutes
Intervals                               : 672
Healthy observation rows                : 1,432,704
```

The persistent root-of-trust lives under Google Drive and downstream notebooks must validate it before consuming the foundation.


#### Section 5 — Synthetic V1 Design Philosophy

##### The environment is not a KPI generator

The design principle is:

> **We are not merely simulating KPIs. We are simulating an observable network whose KPIs emerge from underlying conditions.**

```text
Underlying network conditions
        │
        ├── radio
        ├── load/resource
        ├── processing
        ├── RACH/paging
        ├── N2/SCTP
        ├── F1/E1
        ├── configuration/events
        └── topology/context
                │
                ▼
        Procedure behaviour
                │
         RRC / NG / DRB
                │
                ▼
       standardized PM counters
                │
                ▼
    deterministic KPI derivation
```

##### Three evidence classes

| Evidence class | Meaning | Example |
|---|---|---|
| **Formula evidence** | standardized PMs required to calculate/decompose a KPI | `RRC.ConnEstabAtt.Cause` |
| **Diagnostic evidence** | network observations that can explain degradation | RSRP, SINR, PRB, N2 loss |
| **Context evidence** | scope and chronology used to discriminate hypotheses | peer cells, shared CU, onset after change |

##### Formal benchmark coverage rule

> **A synthetic incident may only enter the formal benchmark if the telemetry environment exposes sufficient observable evidence to distinguish that incident from the other plausible causal hypotheses in scope.**

##### Leakage rule

Agent-visible telemetry must never contain:

```text
root_cause
ground_truth
fault_label
expected_diagnosis
correct_action
benchmark_answer
```

Hidden experiment truth is maintained separately by the benchmark harness.


#### Section 6 — Accessibility KPI and Investigation Scope

Module 5 V1 contains **two formal standardized TS 28.554 RAN Accessibility KPIs**:

1. **Partial DRB Accessibility for UE Services**
2. **Total DRB Accessibility for UE Services**

Those formal KPIs decompose into **seven investigation components/stages**:

| # | Investigation component | Classification |
|---|---|---|
| 1 | Random Access / RACH | supporting pre-RRC diagnostic — **not** a TS 28.554 Accessibility KPI |
| 2 | RRC Connection Establishment | KPI component |
| 3 | UE-Associated Logical NG-Connection Establishment | KPI component |
| 4 | Initial DRB Establishment | KPI component |
| 5 | DRB Establishment | KPI component |
| 6 | RRC Resume / Fallback / Re-establishment | KPI component |
| 7 | DRB Resume | KPI component |

##### Investigation hierarchy

```text
Formal Accessibility KPI
        ↓
Which component degraded?
        ↓
What standardized PM evidence changed?
        ↓
What causal evidence discriminates the mechanism?
        ↓
What topology / chronology supports or excludes hypotheses?
        ↓
Evidence-supported diagnosis
```

The word **component** is used throughout this notebook unless an item is itself a formal standardized KPI.


#### Section 7 — Canonical Evidence Catalogue

##### Standardized PM families — 21

The formal standardized catalogue used by the foundation is:

- `RRC.ConnEstabAtt.Cause`
- `RRC.ConnEstabSucc.Cause`
- `UECNTX.ConnEstabAtt.Cause`
- `UECNTX.ConnEstabSucc.Cause`
- `DRB.InitialEstabAtt.5QI`
- `DRB.InitialEstabSucc.5QI`
- `DRB.EstabAtt.5QI`
- `DRB.EstabSucc.5QI`
- `DRB.EstabAtt.SNSSAI`
- `DRB.EstabSucc.SNSSAI`
- `RRC.ResumeAtt.Cause`
- `RRC.ResumeSucc.Cause`
- `RRC.ResumeFallbackToSetupAtt.cause`
- `RRC.ResumeSuccByFallback.cause`
- `RRC.ReEstabFallbackToSetupAtt`
- `RRC.ReEstabSuccWithoutUeContext`
- `DRB.ResumeAtt.5QI`
- `DRB.ResumeSucc.5QI`
- `RACH.PreambleDedCell`
- `RACH.PreambleACell`
- `RACH.PreambleBCell`

The first 18 are KPI-dependency measurements; the three `RACH.Preamble*` measurements are standardized supporting PM evidence.

##### Operational canonical families — 49

Operational metric names are **canonical synthetic concepts**, not claims that each is a standardized 3GPP PM name. They provide the discriminating network evidence required by the causal design.


##### Radio

- `radio.rsrp_dbm`
- `radio.sinr_db`
- `radio.dl_bler_pct`
- `radio.ul_bler_pct`


##### Resource / load

- `resource.dl_prb_util_pct`
- `resource.ul_prb_util_pct`
- `resource.scheduler_pressure_pct`
- `load.connected_ues`
- `load.active_ues`
- `resource.bearer_admission_pressure_pct`


##### RACH

- `rach.contention_index`
- `rach.access_delay_ms`


##### Paging

- `paging.attempts`
- `paging.responses`
- `paging.load_index`


##### Processing

- `processing.du_cpu_pct`
- `processing.du_queue_depth`
- `processing.cu_cp_cpu_pct`
- `processing.cu_cp_memory_pct`
- `processing.cu_cp_control_queue_depth`
- `processing.cu_cp_control_delay_ms`
- `processing.cu_up_cpu_pct`
- `processing.cu_up_memory_pct`


##### N2 / SCTP

- `transport.n2_util_pct`
- `transport.n2_latency_ms`
- `transport.n2_packet_loss_pct`
- `sctp.association_state`
- `sctp.retransmissions`
- `sctp.heartbeat_failures`


##### AMF-facing boundary

- `boundary.amf_reachable`
- `boundary.amf_peer_state`
- `boundary.amf_overload_indication`


##### F1 / E1 internal interfaces

- `transport.f1_latency_ms`
- `transport.f1_packet_loss_pct`
- `transport.f1_state`
- `transport.e1_latency_ms`
- `transport.e1_state`


##### Context

- `context.ue_context_availability_pct`
- `context.bearer_context_availability_pct`


##### Configuration

- `config.prach_profile_state`
- `config.rrc_profile_state`
- `config.qos_profile_state`
- `config.slice_profile_state`
- `config.resume_profile_state`
- `config.n2_peer_state`


##### Events

- `event.alarm_state`
- `event.interface_alarm_state`
- `event.config_change`
- `event.software_restart`


##### Healthy observation behavior

Of the 49 operational families:

```text
45 continuous / snapshot families
4 event-driven families
```

The healthy baseline intentionally contains **zero event occurrences**. Event channels remain configured and become populated only when a controlled incident legitimately creates an alarm, configuration change, interface event or software restart.


#### Section 8 — Component-by-Component Causal Observability

The following seven subsections preserve the detailed design logic used to build the machine-readable causal-observability configuration. Each component identifies formula evidence, discriminating operational evidence, plausible V1 causal families and the capture conditions that must be satisfied.


##### Component 1 — Random Access / RACH

###### Role

Random Access is the **pre-RRC supporting diagnostic** used to determine whether Accessibility degradation begins before RRC establishment.

It is **not** represented as a standardized TS 28.554 Accessibility KPI.

###### Formula / standardized evidence already frozen

```text
RACH.PreambleDedCell
RACH.PreambleACell
RACH.PreambleBCell
```

Scope: `NRCellDU`

###### Diagnostic evidence required for causal discrimination

| Evidence | Why it matters |
|---|---|
| RSRP | weak coverage can produce repeated/failed access attempts |
| SINR / interference indication | poor radio quality can impair access even with acceptable RSRP |
| RACH preamble distribution | distinguishes normal access load from abnormal contention/repetition |
| RACH access-delay / contention indicator | useful operational evidence of access stress |
| UL PRB/resource pressure | supports congestion/resource-pressure hypothesis |
| connected/active users | provides demand context |
| DU CPU / processing pressure | separates radio-access load from DU processing saturation |
| PRACH configuration state/change | supports misconfiguration hypothesis |
| alarms/events | corroborates DU/radio/interface fault |
| peer-cell comparison | shows whether symptom is cell-local, site-wide or broader |

###### Plausible V1 fault families

- poor coverage
- uplink interference / poor SINR
- RACH contention / excessive access load
- DU processing pressure
- PRACH configuration mismatch/change
- radio/DU alarm condition

###### Foundation capture checklist

- [ ] 3 standardized RACH PM families generated
- [ ] NRCellDU entity scope preserved
- [ ] RSRP generated
- [ ] SINR generated
- [ ] UL PRB/resource pressure generated
- [ ] connected/active user load generated
- [ ] DU processing metric generated
- [ ] RACH operational contention/access-delay evidence available if included in V1
- [ ] configuration state/change evidence model exists
- [ ] alarms/events model exists
- [ ] peer-cell/topology context available
- [ ] no fabricated standardized RACH Success KPI created


##### Component 2 — RRC Connection Establishment Success

###### Formula evidence

```text
RRC.ConnEstabAtt.Cause
RRC.ConnEstabSucc.Cause
```

Important dimensionality: **Cause**

The synthetic environment should preserve at least a compact cause subset that allows `mo-Signalling` exclusion logic to be exercised later.

###### Diagnostic / causal evidence

```text
RRC Connection Success
      │
      ├── FORMULA COUNTERS
      │     ├─ RRC.ConnEstabAtt.Cause
      │     └─ RRC.ConnEstabSucc.Cause
      │
      └── DIAGNOSTIC / CAUSAL EVIDENCE
            ├─ RACH behaviour
            ├─ RSRP / SINR
            ├─ DL / UL PRB utilisation
            ├─ connected / active users
            ├─ paging behaviour for MT-access cases
            ├─ DU / CU-CP processing pressure
            ├─ alarms / events
            └─ configuration / recent change
```

###### Plausible V1 fault families

1. pre-RRC / RACH impairment
2. poor coverage
3. interference / poor SINR
4. radio resource congestion
5. DU processing overload
6. CU-CP processing pressure
7. RRC-related configuration change
8. radio/software restart or alarm condition
9. paging-related MT-access issue for applicable cause patterns

###### Example discriminating patterns

**Coverage issue**

```text
RRC SR       ↓
RSRP         ↓
SINR         ↓
RACH stress  ↑
PRB          normal
processing   normal
```

**Radio congestion**

```text
RRC SR       ↓
RSRP/SINR    healthy
PRB          ↑↑
users        ↑
RACH load    ↑
processing   possibly ↑
```

**Configuration issue**

```text
RRC SR       ↓
radio        healthy
PRB          normal
processing   normal
recent change YES
alarm/event  correlated
```

###### Foundation capture checklist

- [ ] RRC attempts by cause generated
- [ ] RRC successes by cause generated
- [ ] RACH PMs generated
- [ ] RSRP generated
- [ ] SINR generated
- [ ] DL PRB generated
- [ ] UL PRB generated
- [ ] connected/active users generated
- [ ] paging evidence available for MT-access investigations
- [ ] DU processing evidence generated
- [ ] CU-CP processing evidence generated
- [ ] configuration/change evidence model available
- [ ] alarm/event evidence model available
- [ ] topology/time context available


##### Component 3 — UE-Associated Logical NG-Connection Establishment

###### Formula evidence

Canonical TS 28.552 PM catalogue names:

```text
UECNTX.ConnEstabAtt.Cause
UECNTX.ConnEstabSucc.Cause
```

The TS 28.554 formula spelling using `UECNTXT.*` remains an explicit alias and should not replace the canonical PM catalogue name.

Scope: `NRCellCU`

###### Causal-observability structure

```text
UE-ASSOCIATED LOGICAL NG-CONNECTION SUCCESS
                         │
             ┌───────────┴───────────┐
             ▼                       ▼
      FORMULA EVIDENCE         DIAGNOSTIC EVIDENCE
             │                       │
     ┌───────┴────────┐       ┌──────┼──────────────────────┐
     ▼                ▼       ▼      ▼                      ▼
NG Attempts        NG Success  RAN   N2 / SCTP            AMF-facing
by Cause           by Cause   CU-CP  Transport             Boundary
     │                │       │      │                      │
UECNTX.           UECNTX.     CPU    utilisation           reachability
ConnEstabAtt      ConnEstabSucc memory latency             peer/path state
.Cause             .Cause     queue  packet loss           overload indication
                                  │   SCTP association
                                  │   retransmissions
                                  │
                                  ├──── ALARMS / EVENTS
                                  └──── CONFIG / CHANGE
```

###### Upstream / exclusion evidence

Before concluding an NG-stage problem, the agent must be able to establish whether the UE successfully reached this stage:

- RRC establishment health
- RACH health
- radio health
- radio/resource pressure

###### Diagnostic evidence required

| Evidence family | Candidate observations |
|---|---|
| CU-CP processing | CPU, memory, control queue depth, processing delay |
| N2 transport | utilisation, latency, packet loss |
| SCTP | association state, retransmissions, heartbeat/path failures |
| AMF-facing boundary | reachability, peer state, overload indication |
| Configuration | AMF peer endpoint state, N2/SCTP config, recent change |
| Alarms/events | N2 alarm, SCTP alarm, CU restart/software event |
| Topology | affected cells/sites, shared CU domain, shared N2/AMF path |

###### Plausible V1 fault families

1. upstream RRC degradation — not an NG root cause
2. CU-CP processing overload
3. N2 transport congestion
4. N2 packet loss / latency impairment
5. SCTP association instability
6. N2/AMF peer configuration problem
7. CU software/restart event
8. AMF-facing / beyond-RAN boundary condition

###### Example discriminating patterns

**N2 congestion**

```text
NG SR                   ↓↓↓
RRC                     healthy
radio                    healthy
N2 utilisation           ↑↑
N2 latency               ↑
N2 packet loss           ↑
SCTP retransmissions     ↑
SCTP association         UP
CU-CP CPU                normal
```

**CU-CP overload**

```text
NG SR                   ↓↓
RRC                     healthy / mild impact
N2 utilisation          normal
N2 loss                 normal
SCTP                    stable
CU-CP CPU               ↑↑
control queue depth     ↑↑
processing delay        ↑↑
```

**AMF-facing boundary**

```text
NG SR                   ↓↓↓
RRC                     healthy
radio/resource          healthy
CU-CP                   healthy
N2 transport            healthy
AMF reachability        intermittent
shared AMF-facing path  affected
```

The diagnosis should remain **boundary-scoped** unless evidence actually proves an internal Core fault.

###### Foundation capture checklist

- [ ] NG attempts by cause generated
- [ ] NG successes by cause generated
- [ ] upstream RRC evidence generated
- [ ] RACH/radio/resource exclusion evidence generated
- [ ] CU-CP CPU generated
- [ ] CU-CP memory/queue/processing delay represented
- [ ] N2 utilisation generated
- [ ] N2 latency generated
- [ ] N2 packet loss generated
- [ ] SCTP association state generated
- [ ] SCTP retransmission/heartbeat evidence generated
- [ ] AMF-facing reachability/peer-state evidence represented
- [ ] N2/SCTP configuration state/change represented
- [ ] N2/SCTP/CU alarms/events represented
- [ ] shared-CU/shared-path topology context represented


##### Component 4 — Initial DRB Establishment Success

###### Formula evidence

```text
DRB.InitialEstabAtt.5QI
DRB.InitialEstabSucc.5QI
```

Scope: `NRCellCU`

Important dimensionality: **5QI**

###### Investigation logic

```text
Initial DRB SR degraded
        │
        ├── Did RRC succeed?
        ├── Did NG signalling succeed?
        │
        └── If upstream stages are healthy:
                 │
                 ├── radio quality?
                 ├── radio capacity?
                 ├── bearer/admission resources?
                 ├── CU-UP / DU processing?
                 ├── internal transport/state?
                 ├── 5QI-specific configuration?
                 └── alarms / recent changes?
```

###### Diagnostic evidence required

- RRC and NG prerequisite health
- RSRP / SINR
- DL and UL PRB utilisation
- connected / active users
- DL/UL BLER where retained in V1
- scheduler/admission pressure indicator
- DU processing pressure
- CU-UP processing/resource state if modeled
- F1/E1 internal path state where needed to distinguish architecture faults
- 5QI-specific distribution
- QoS / bearer configuration state
- alarms/events
- configuration/software change history
- peer-cell comparison

###### Plausible V1 fault families

1. upstream RRC/NG degradation
2. poor radio quality
3. radio resource congestion
4. bearer/admission resource pressure
5. DU processing overload
6. CU-UP / internal user-plane preparation issue
7. 5QI/QoS configuration mismatch
8. F1/E1/internal interface degradation where modeled
9. software/configuration event

###### Foundation capture checklist

- [ ] Initial DRB attempts by 5QI generated
- [ ] Initial DRB successes by 5QI generated
- [ ] RRC prerequisite evidence generated
- [ ] NG prerequisite evidence generated
- [ ] RSRP/SINR generated
- [ ] DL/UL PRB generated
- [ ] connected/active users generated
- [ ] BLER retained if part of V1 fault coverage
- [ ] scheduler/admission pressure concept represented
- [ ] DU processing evidence generated
- [ ] CU-UP/resource evidence represented if included
- [ ] F1/E1/internal interface evidence represented where needed
- [ ] 5QI/QoS configuration evidence represented
- [ ] alarms/events/config changes represented


##### Component 5 — DRB Establishment Success

###### Formula evidence

```text
DRB.EstabAtt.5QI
DRB.EstabSucc.5QI

DRB.EstabAtt.SNSSAI
DRB.EstabSucc.SNSSAI
```

Scope: `NRCellCU`

Important dimensions:

- **5QI**
- **S-NSSAI**

###### Why this component needs richer causal evidence

A total DRB establishment degradation may be:

- general across all 5QIs
- concentrated in one QoS class
- concentrated in one S-NSSAI
- driven by radio resource pressure
- driven by admission/scheduler behaviour
- driven by configuration
- driven by processing or interface state

Therefore aggregate DRB SR alone is insufficient.

###### Diagnostic evidence required

| Area | Evidence |
|---|---|
| Upstream | RRC and NG health |
| Radio | RSRP, SINR, BLER |
| Capacity | DL/UL PRB, active users |
| Admission/scheduler | resource/admission pressure |
| QoS | 5QI-specific attempts/successes and configuration |
| Slice | S-NSSAI-specific attempts/successes and configuration |
| Processing | DU and CU-UP resource state |
| Internal interface | F1/E1 state/latency/loss where relevant |
| Events | bearer-related alarms, restarts |
| Configuration | QoS/slice/bearer parameter changes |
| Context | peer-cell and cross-slice comparison |

###### Plausible V1 fault families

1. general radio congestion
2. poor radio quality
3. 5QI-specific configuration or admission issue
4. S-NSSAI-specific configuration/resource issue
5. DU processing overload
6. CU-UP/resource preparation issue
7. F1/E1/internal interface issue where modeled
8. configuration/software change

###### Example discriminating pattern — slice-specific issue

```text
Total DRB SR               mildly ↓
5QI behaviour              mostly healthy
S-NSSAI 1-010203           ↓↓↓
radio                      healthy
PRB                        normal
processing                 normal
slice config change        YES
```

This should not be diagnosed as generic radio congestion.

###### Foundation capture checklist

- [ ] DRB attempts by 5QI generated
- [ ] DRB successes by 5QI generated
- [ ] DRB attempts by S-NSSAI generated
- [ ] DRB successes by S-NSSAI generated
- [ ] RRC/NG prerequisite evidence generated
- [ ] radio quality/capacity evidence generated
- [ ] BLER represented if retained
- [ ] admission/scheduler pressure represented
- [ ] DU/CU-UP processing represented
- [ ] 5QI configuration context represented
- [ ] S-NSSAI configuration context represented
- [ ] alarms/events represented
- [ ] peer-cell / peer-slice context represented


##### Component 6 — RRC Resume / Fallback / Re-establishment

###### Standardized formula-related evidence

```text
RRC.ResumeAtt.Cause
RRC.ResumeSucc.Cause

RRC.ResumeSuccByFallback.cause
RRC.ResumeFallbackToSetupAtt.cause

RRC.ReEstabSuccWithoutUeContext
RRC.ReEstabFallbackToSetupAtt
```

The first two use the TS 28.552 canonical uppercase `.Cause`; the fallback measurements retain the standardized lower-case `.cause` spelling already frozen in the KPI knowledge supplement.

###### Why this component is structurally different

Resume is not just a simple success/attempt pair. The agent must distinguish:

- direct Resume success
- fallback to Setup
- re-establishment behaviour
- loss/unavailability of retained UE context
- radio-access problems during Resume
- processing/configuration issues
- software/restart events that affect context retention

###### Diagnostic evidence required

- RACH behaviour
- RSRP / SINR
- PRB utilisation and active users
- CU-CP CPU/memory/queue pressure
- UE context / inactive-context availability indicator in the synthetic model
- Resume/fallback/re-establishment distribution
- paging evidence where the scenario involves MT activation
- configuration state for inactivity/Resume-related parameters
- CU restart/software event
- alarms/events
- peer-cell/shared-CU correlation

###### Plausible V1 fault families

1. radio degradation during Resume
2. RACH/access issue during Resume path
3. CU-CP processing pressure
4. lost/unavailable UE context
5. abnormal fallback-to-Setup behaviour
6. elevated re-establishment behaviour
7. Resume-related configuration issue
8. CU restart/software event affecting context retention
9. paging-related activation issue where applicable

###### Example pattern — context retention / restart issue

```text
RRC Setup SR                       healthy
RRC Resume direct success          ↓↓↓
ResumeFallbackToSetupAtt           ↑↑
ResumeSuccByFallback               ↑
ReEstab*                           ↑
radio                              healthy
PRB                                normal
CU restart event                   correlated
```

###### Foundation capture checklist

- [ ] Resume attempts by cause generated
- [ ] direct Resume successes by cause generated
- [ ] Resume fallback attempts generated
- [ ] Resume success-by-fallback generated
- [ ] re-establishment-without-context generated
- [ ] re-establishment fallback-to-Setup generated
- [ ] RACH/radio/resource evidence generated
- [ ] CU-CP processing evidence generated
- [ ] UE-context availability concept represented
- [ ] paging evidence represented where applicable
- [ ] Resume-related configuration context represented
- [ ] restart/software/alarm events represented
- [ ] shared-CU topology context represented


##### Component 7 — DRB Resume Success

###### Formula evidence

```text
DRB.ResumeAtt.5QI
DRB.ResumeSucc.5QI
```

Scope: `NRCellCU`

Important dimensionality: **5QI**

###### Investigation logic

DRB Resume sits downstream of the RRC Resume path.

Therefore a degraded DRB Resume success ratio should first be decomposed into:

```text
DRB Resume degraded
        │
        ├── Was RRC Resume healthy?
        │
        └── If RRC Resume is healthy:
                 │
                 ├── radio quality / capacity
                 ├── bearer-context availability
                 ├── CU-UP / DU processing
                 ├── QoS / 5QI-specific behaviour
                 ├── internal interface state
                 ├── alarms
                 └── config/software changes
```

###### Diagnostic evidence required

- RRC Resume/fallback/re-establishment evidence
- RSRP / SINR
- DL/UL PRB
- active/connected users
- BLER where retained
- DU processing
- CU-UP processing/resource state
- bearer-context availability
- F1/E1/internal interface state where required
- 5QI-specific behaviour
- alarms/events
- configuration/software change history

###### Plausible V1 fault families

1. upstream RRC Resume degradation
2. radio quality issue
3. radio/resource congestion
4. bearer-context loss/unavailability
5. CU-UP/DU processing pressure
6. 5QI-specific Resume issue
7. internal interface impairment
8. configuration/software change

###### Foundation capture checklist

- [ ] DRB Resume attempts by 5QI generated
- [ ] DRB Resume successes by 5QI generated
- [ ] RRC Resume prerequisite evidence generated
- [ ] fallback/re-establishment evidence generated
- [ ] radio quality/capacity evidence generated
- [ ] DU processing evidence generated
- [ ] CU-UP/resource evidence represented
- [ ] bearer-context availability represented
- [ ] 5QI-specific context represented
- [ ] F1/E1/internal interface evidence represented where needed
- [ ] alarms/events/config changes represented


#### Section 9 — Cross-Component Observability Matrix


Legend:

- **P** = primary diagnostic evidence
- **S** = supporting / exclusion evidence
- **C** = context evidence
- blank = normally not required for the component

| Evidence family | RACH | RRC Setup | NG Signalling | Initial DRB | DRB Estab | RRC Resume | DRB Resume |
|---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| Standard formula PM | S | P | P | P | P | P | P |
| RACH evidence | P | P | S | S | S | P | S |
| RSRP | P | P | S | P | P | P | P |
| SINR | P | P | S | P | P | P | P |
| DL PRB | S | P | S | P | P | P | P |
| UL PRB | P | P | S | P | P | P | P |
| Connected/active users | S | P | S | P | P | P | P |
| Paging |  | S | S |  |  | S |  |
| CU-CP processing |  | S | P | S | S | P | S |
| DU processing | P | P | S | P | P | S | P |
| CU-UP processing |  |  |  | P | P |  | P |
| N2 transport |  |  | P |  |  |  |  |
| SCTP |  |  | P |  |  |  |  |
| AMF-facing boundary |  |  | P |  |  |  |  |
| F1/E1/internal interface |  | S | S | S/P | S/P | S | S/P |
| Alarm/event | P | P | P | P | P | P | P |
| Configuration/change | P | P | P | P | P | P | P |
| Topology correlation | C | C | C | C | C | C | C |
| Chronology | C | C | C | C | C | C | C |

##### Interpretation

This matrix is not a causal truth table. It is a **telemetry completeness map**.

For example:

- N2 transport is primary for NG signalling but usually irrelevant to a pure pre-RRC RACH fault.
- RACH evidence is primary for RRC access and supporting/exclusion evidence for later stages.
- Configuration, alarms, topology and chronology are cross-cutting because they help distinguish mechanisms even when scalar KPI symptoms look similar.


#### Section 10 — Fault-Family and Incident Benchmark Design

The current causal-observability configuration contains **37 unique fault-family concepts** across **58 component/fault-family mappings**.

##### Declared V1 causal concepts

- `ADMISSION_RESOURCE_PRESSURE`
- `AMF_FACING_BOUNDARY`
- `BEARER_CONTEXT_UNAVAILABLE`
- `CU_CP_PROCESSING_PRESSURE`
- `CU_RESTART_SOFTWARE_EVENT`
- `CU_UP_RESOURCE_PRESSURE`
- `DU_PROCESSING_PRESSURE`
- `FALLBACK_ABNORMALITY`
- `FIVE_QI_SPECIFIC`
- `INTERNAL_INTERFACE`
- `N2_CONGESTION`
- `N2_PACKET_LOSS_LATENCY`
- `N2_PEER_CONFIGURATION`
- `PAGING_MT_ACCESS`
- `POOR_COVERAGE`
- `POOR_RADIO_QUALITY`
- `PRACH_CONFIGURATION`
- `QOS_CONFIGURATION`
- `RACH_CONTENTION`
- `RACH_RESUME_ACCESS`
- `RADIO_DU_ALARM`
- `RADIO_INTERFERENCE`
- `RADIO_RESOURCE_CONGESTION`
- `RADIO_RESUME_DEGRADATION`
- `REESTABLISHMENT_ABNORMALITY`
- `RESUME_CONFIGURATION`
- `RRC_CONFIGURATION`
- `SCTP_INSTABILITY`
- `SNSSAI_SPECIFIC`
- `SOFTWARE_CONFIGURATION_EVENT`
- `SOFTWARE_EVENT`
- `UE_CONTEXT_UNAVAILABLE`
- `UL_INTERFERENCE`
- `UPSTREAM_RACH`
- `UPSTREAM_RRC`
- `UPSTREAM_RRC_NG`
- `UPSTREAM_RRC_RESUME`

##### Incident design contract

Every formal incident must define:

1. **target component and fault family**;
2. **observable signature** that the synthetic network will produce;
3. **nearest competing hypotheses** the agent must distinguish;
4. **required supporting/exclusion evidence**;
5. **expected non-causal distractors**;
6. **topology scope** — cell, site, DU, CU domain, path or peer boundary;
7. **chronology** — onset, persistence and recovery behavior;
8. **hidden truth** stored outside agent-visible telemetry;
9. **success criteria** for diagnosis and any recommended action.

> **Do not inject a fault family until its observable signature, nearest competing hypothesis, required evidence and expected non-causal distractors have been explicitly defined.**

##### Why this matters

The benchmark should not reduce to:

```text
KPI degraded
    ↓
one pre-programmed cause
    ↓
obvious answer
```

Instead:

```text
KPI degraded
    ↓
multiple plausible hypotheses
    ↓
formula evidence
+ diagnostic evidence
+ context / chronology
    ↓
agent investigation
    ↓
evidence-supported diagnosis
```


##### Component-level initial fault-family plan

The next runtime design should convert this notebook into explicit configuration objects.

A sensible initial V1 fault-family set is:

| Component | Initial synthetic fault families |
|---|---|
| RACH | coverage, interference, contention/load, DU pressure, PRACH config |
| RRC Setup | RACH/upstream, coverage, interference, PRB congestion, processing, config/change |
| NG Signalling | CU-CP overload, N2 congestion, N2 loss/latency, SCTP instability, peer config, AMF-facing boundary |
| Initial DRB | upstream NG, radio quality, capacity/admission, DU/CU-UP processing, QoS config |
| DRB Establishment | radio congestion, 5QI issue, S-NSSAI issue, processing, internal interface, config |
| RRC Resume | radio/RACH, CU-CP pressure, context loss, fallback/re-establishment, restart/config |
| DRB Resume | upstream Resume, radio/capacity, bearer context, CU-UP/DU processing, 5QI/internal interface |

###### Freeze rule before incident generation

> Do not inject a fault family until its observable signature, nearest competing hypothesis, required evidence and expected non-causal distractors have been explicitly defined.

This keeps the synthetic benchmark diagnosable rather than artificially solvable.


#### Section 11 — Healthy Baseline and KPI Validation Plan

The completed healthy baseline is the **normal operating reference**, not an incident dataset.

##### Healthy baseline coverage

```text
7 days
672 × 15-minute intervals
30 logical NR cells
21 standardized PM families
45 continuously observed operational families
4 configured event-driven families with zero healthy events
1,432,704 total observation rows
```

The generator uses deterministic seeds and shared latent radio/load/processing/transport state so standardized PM counters and operational evidence remain causally correlated.

##### What is deliberately not persisted as an input variable

The two formal Accessibility KPI values are **not sampled or stored as independently generated truth**.

```text
Healthy network state
      ↓
standardized attempts / successes
      ↓
M05_02 deterministic formula engine
      ↓
Partial DRB Accessibility
Total DRB Accessibility
```

##### M05_02 validation objectives

The next notebook should:

1. validate the root manifest and restore frozen artifacts from Drive;
2. load all seven healthy Parquet partitions;
3. reconstruct dimensions such as Cause, 5QI and S-NSSAI correctly;
4. derive the two formal Accessibility KPIs using the frozen standardized dependencies;
5. verify expected healthy behavior across time, cell, site and CU-domain views;
6. validate internal consistency between prerequisite components;
7. identify any synthetic-generation anomalies before incident injection;
8. freeze a healthy KPI validation artifact for downstream benchmarks.

The healthy baseline should be treated as the **control group** for every later incident.


#### Section 12 — Independent Simulator Reference Validation

The custom Python generator is intentionally a **controlled synthetic telemetry environment**, not an industry-standard 5G network simulator. To avoid treating it as self-validating, Module 5 will add an independent reference notebook using **NVIDIA Sionna SYS** for selected radio/system-level behavior.

##### What the comparison can validate

Sionna is useful as an independent reference for relationships such as:

```text
SINR distribution
SINR ↔ BLER behavior
offered traffic ↔ resource pressure
load ↔ throughput
interference/load ↔ radio-quality degradation
```

##### What it should not be used to claim

Sionna does not independently reproduce the complete Module 5 Accessibility observation catalogue. It should not be presented as validation of:

```text
RRC.ConnEstabAtt.Cause
UECNTX.ConnEstabSucc.Cause
DRB.ResumeSucc.5QI
CU-CP queue depth
SCTP association state
AMF-facing reachability
configuration events
```

##### Reference-validation question

> **Do the radio/resource relationships engineered into Synthetic V1 behave consistently with an independently developed 5G system-level simulator under comparable broad conditions?**

The purpose is **external sanity reference**, not exact dataset reproduction.

##### Planned notebook

`M05_03_Sionna_Radio_Reference_Validation_v1.ipynb`

This should remain separate from the formal agent benchmark so simulator-specific implementation details do not leak into the agent's evidence environment.


#### Section 13 — Synthetic V1 → Executable V2 → Live Alignment


The long-term interface should remain:

```text
                    CANONICAL TELEMETRY ONTOLOGY
                               │
          ┌────────────────────┼────────────────────┐
          ▼                    ▼                    ▼
   SYNTHETIC V1          EXECUTABLE V2          LIVE/COLLECTED
   generator             simulator/emulator      PM/FM/CM source
          │                    │                    │
          └────────────────────┼────────────────────┘
                               ▼
                         TELEMETRY MCP
                               ▼
                             AGENT
```

##### Alignment rule

The **source implementation may change**, but the agent-facing evidence concept should remain stable wherever possible.

Examples:

```text
Synthetic V1 concept
resource.dl_prb_util_pct
        ↓
Executable V2 metric adapter
        ↓
Live PM/vendor adapter
        ↓
Canonical Telemetry MCP evidence
```

```text
Synthetic V1 concept
transport.n2_packet_loss_pct
        ↓
Executable V2 N2 measurement
        ↓
Live transport/OAM source
        ↓
Canonical Telemetry MCP evidence
```

##### Why this matters

Without this alignment, Module 5 could demonstrate good synthetic-agent behaviour that does not transfer to an executing or collected network environment.

The causal-observability reference therefore serves two roles:

1. **foundation generation checklist**
2. **V2/live telemetry mapping specification**


##### Executable V2 intent

Synthetic V1 provides controlled causal experimentation. Executable V2 should introduce actual protocol/software behavior through an executable 5G environment such as OAI/srsRAN/UERANSIM/Open5GS or another qualified stack without changing the agent-facing telemetry ontology unnecessarily.

```text
SYNTHETIC V1
engineered network state
        ↓
Can the agent reason correctly?

EXECUTABLE V2
actual protocol/software procedures
        ↓
Does the capability transfer?

LIVE / COLLECTED
PM / FM / CM / logs / platform data
        ↓
Can the same ontology and investigation architecture map to operational evidence?
```

The **source implementation may change; the canonical evidence concept should remain stable wherever possible.**


#### Section 14 — MCP and Agent Investigation Plan

##### Knowledge plane

Existing knowledge architectures from Modules 2–4 are reused rather than rebuilt:

```text
Semantic RAG MCP
    └── search_semantic_telecom_knowledge()

Knowledge MCP
    └── search_telecom_knowledge()
```

Their retrieval mechanisms and provenance remain distinguishable.

##### Telemetry MCP

The Telemetry MCP should expose deterministic hierarchical access rather than dumping the entire dataset into the context window.

Illustrative interface:

```text
get_kpis()
get_kpi_components()
get_measurements()
get_radio_state()
get_resource_state()
get_transport_state()
get_processing_state()
get_alarms()
get_events()
get_configuration()
get_topology()
```

##### Investigation hierarchy

```text
Level 1
What KPI/category is degraded?
        ↓
Level 2
Which Accessibility component is driving it?
        ↓
Level 3
Which standardized PM evidence changed?
        ↓
Level 4
Which causal evidence discriminates the mechanism?
        ↓
Level 5
What topology / chronology confirms or excludes the hypothesis?
```

##### LangGraph state concept

The evolving `AgentState` should preserve at least:

```text
incident
current scope
evidence collected
knowledge retrieved
current hypotheses
tool-call history
ML evidence
investigation depth
round count
errors / unavailable tools
evidence sufficiency
diagnosis
recommended action
policy decision
action result
verification result
stop condition
```

The LLM chooses **what evidence to seek next**; deterministic tools remain responsible for returning network truth.


#### Section 15 — ML, Policy, Action and Verification Plan

##### Specialist ML

ML is a **probabilistic evidence source**, not an oracle.

Planned ML MCP capabilities may include:

```text
detect_anomaly()
predict_fault()
estimate_fault_likelihood()
```

The agent must be able to:

- request ML only when it adds value;
- compare predictions against deterministic telemetry;
- reject or downgrade contradictory ML evidence;
- explain why ML evidence was or was not used.

##### Policy

Action authorization remains deterministic:

```text
Recommended action
       ↓
Policy engine
  ┌────┼─────┐
ALLOW APPROVAL DENY
  │      │
  └──┬───┘
     ↓
Action MCP
```

The LLM must not have unrestricted operational authority.

##### Verification

A successful action is not assumed to have fixed the incident.

```text
Action executed
      ↓
Observe network again
      ↓
Recalculate affected KPI/components
      ↓
Did evidence improve?
   ┌────┴────┐
  YES       NO
   │         │
Resolve   Continue / rollback / escalate
```

> **Verification closes the control loop and prevents “action executed” from being treated as equivalent to “problem solved.”**


#### Section 16 — Evaluation Framework

Module 5 evaluates **trajectory quality**, not only final-answer correctness.

##### Core evaluation dimensions

| Dimension | Example measure |
|---|---|
| Diagnostic correctness | correct fault family / correct boundary-safe diagnosis |
| Evidence grounding | diagnosis supported by observed evidence |
| Evidence sufficiency | enough evidence collected before conclusion |
| Tool selection quality | relevant tools chosen at appropriate depth |
| Tool efficiency | unnecessary calls / redundant retrieval |
| Investigation depth | how far hierarchy was traversed |
| Premature diagnosis | conclusion before discriminating evidence |
| Knowledge efficiency | RAG/MCP invoked only when useful |
| ML handling | prediction validated rather than blindly accepted |
| Contradiction handling | recovery from misleading/conflicting evidence |
| Failure recovery | behavior when tool/service unavailable |
| Latency | total and per-tool time |
| Token consumption | model cost/efficiency proxy |
| Policy compliance | no unauthorized action |
| Action effectiveness | post-action state actually improves |
| Verification completeness | outcome observed before closure |

##### Trajectory example

```text
Agent A
3 justified tool calls
correct evidence
correct RCA

Agent B
17 calls
irrelevant retrieval
accepts misleading ML evidence
eventually reaches same RCA
```

Both may have the same final answer, but they are **not equally capable agents**.

LangSmith provides interactive trajectory observability; formal evaluation artifacts remain independent so the experiment is reproducible without LangSmith.


#### Section 17 — Master Readiness Checklist


Before executing the healthy telemetry generator, answer **YES** to each applicable item.

##### A. Standardized formula evidence

- [ ] All 18 standardized KPI-dependency PM families from the standardized observation contract are generated
- [ ] All 3 standardized supporting RACH PM families are generated
- [ ] Cause dimensionality is preserved where required
- [ ] 5QI dimensionality is preserved where required
- [ ] S-NSSAI dimensionality is preserved where required
- [ ] TS 28.552 canonical naming and the KPI knowledge supplement aliases remain intact
- [ ] no fabricated standardized KPI or PM name is introduced

##### B. Radio and access evidence

- [ ] RSRP
- [ ] SINR
- [ ] DL PRB utilisation
- [ ] UL PRB utilisation
- [ ] connected users
- [ ] active users
- [ ] RACH supporting PM
- [ ] RACH operational contention/access-delay evidence if part of V1
- [ ] paging evidence for applicable MT-access investigations
- [ ] BLER if needed by selected V1 fault families

##### C. Processing evidence

- [ ] DU CPU / processing pressure
- [ ] CU-CP CPU
- [ ] CU-CP memory / queue / processing delay
- [ ] CU-UP processing/resource state if used by DRB fault families
- [ ] processing evidence has hierarchy consistent with the causal topology

##### D. Transport and interface evidence

- [ ] N2 utilisation
- [ ] N2 latency
- [ ] N2 packet loss
- [ ] SCTP association state
- [ ] SCTP retransmission / heartbeat evidence
- [ ] AMF-facing reachability/peer state
- [ ] F1/E1/internal interface evidence where required by selected fault families

##### E. Configuration, events and chronology

- [ ] configuration state model
- [ ] configuration-change event model
- [ ] alarm/event model
- [ ] software restart/change event model
- [ ] onset/change timestamps available
- [ ] agent-visible telemetry does not include root-cause labels

##### F. Topology and correlation

- [ ] logical cell identity
- [ ] NRCellCU identity
- [ ] NRCellDU identity
- [ ] site identity
- [ ] DU identity
- [ ] CU domain identity
- [ ] common transport/peer-path identity where modeled
- [ ] peer-cell comparison possible
- [ ] site-wide correlation possible
- [ ] shared-CU correlation possible

##### G. Experimental safety / leakage

- [ ] healthy baseline contains no fault labels
- [ ] healthy baseline contains no expected diagnosis
- [ ] hidden incident truth is stored separately from agent-visible telemetry
- [ ] synthetic generator parameters are not exposed as diagnosis hints
- [ ] operational synthetic thresholds are not mislabelled as 3GPP standards
- [ ] ML inference remains a separate evidence layer

##### H. Causal coverage

For every fault family intended for the formal benchmark:

- [ ] the fault changes at least one relevant observable
- [ ] there is sufficient evidence to distinguish it from its nearest competing hypothesis
- [ ] there is at least some normal variability/distractor evidence
- [ ] the agent is not forced to infer an unobservable hidden cause
- [ ] the same logical evidence concept can be mapped later to V2/live telemetry


##### Benchmark-readiness additions

Before incident generation begins:

- [ ] healthy KPI derivation has passed;
- [ ] healthy time-series behavior has no unexplained synthetic artifacts;
- [ ] external Sionna reference checks are documented for selected radio/resource relationships;
- [ ] each chosen incident has a nearest competing hypothesis;
- [ ] each incident has both positive evidence and exclusion evidence;
- [ ] hidden truth is isolated from agent-visible data;
- [ ] incident overlays preserve the original healthy baseline;
- [ ] no incident is admitted solely because its injected variable name reveals its diagnosis;
- [ ] all benchmark scenario IDs, seeds and artifact hashes are frozen before formal agent evaluation.


#### Section 18 — Provenance, Reproducibility and Safety Rules

##### Semantic identity versus file identity

```text
Semantic SHA
    → identifies engineering meaning

Byte SHA
    → identifies one exact serialized file instance
```

Generated JSON artifacts may differ byte-for-byte across executions because of non-semantic metadata such as `created_utc`. Such differences are acceptable **only if the frozen semantic identity is unchanged**.

##### Persistence model

```text
Generate / validate under /content
          ↓
freeze local artifact
          ↓
copy through Drive staging
          ↓
verify persisted instance
          ↓
retain immutable Drive artifact
```

Persistent formal artifacts must never be silently overwritten by semantically different content.

##### Root-of-trust rule

Downstream notebooks should begin with:

```text
Mount Google Drive
      ↓
load module5_foundation_manifest_v1.json
      ↓
validate root semantic SHA
      ↓
validate required child artifacts
      ↓
restore only what is needed
```

They should **not rerun the foundation build simply to reconstruct runtime objects**.

##### Truth separation

```text
Agent-visible
------------
telemetry
knowledge
ML predictions
tool results

Hidden benchmark harness
------------------------
injected fault
ground-truth onset
expected diagnosis
expected policy behavior
scoring reference
```

This separation is mandatory for a meaningful agentic evaluation.


#### Section 19 — Current Status and Immediate Next Step

##### Completed

- canonical 3GPP-aligned KPI/telemetry foundation;
- 5G RAN Accessibility graph V1 → V1.1 → V1.2;
- exact KPI knowledge supplement;
- unified observation contract V1.1;
- causal synthetic topology V1.1;
- 70-family causal-observability configuration;
- 132 observation entities;
- 7-day / 1,432,704-row healthy telemetry baseline;
- Google Drive persistence;
- Module 5 root-of-trust foundation manifest.

##### Not yet started in the formal execution sequence

- deterministic derivation of Partial and Total DRB Accessibility from the healthy PM data;
- healthy KPI validation;
- Sionna reference validation;
- controlled incident overlays;
- hidden benchmark truth;
- Telemetry MCP;
- specialist ML MCP;
- LangGraph agent investigation;
- Action MCP/policy loop;
- formal Module 5 evaluation.

##### Immediate next notebook

`M05_02_Runtime_Restore_KPI_Derivation_and_Healthy_Validation_v1.ipynb`

Its first principle is:

> **Restore the frozen foundation; do not rebuild it.**

The output of M05_02 becomes the validated control baseline required before any synthetic incident is admitted into the benchmark.


#### Appendix A — Reference Configuration Skeleton

The following historical skeleton documents how the causal-observability design was first represented before Cell 09 converted it into the formal runtime configuration. It is retained as a **reference only** and should not be executed as a competing source of truth.


```python
# ============================================================
# OPTIONAL REFERENCE CONFIG SKELETON
# ============================================================
#
# This cell is documentation support only.
# It is not required by the Module 5 runtime notebook.
#
# The structure can later be translated into the formal
# synthetic-telemetry configuration after the checklist
# has been reviewed and frozen.
# ============================================================

ACCESSIBILITY_OBSERVABILITY_COMPONENTS_V1 = {

    "rach_access": {
        "classification": "SUPPORTING_PRE_RRC_DIAGNOSTIC",
        "formula_measurements": [
            "RACH.PreambleDedCell",
            "RACH.PreambleACell",
            "RACH.PreambleBCell",
        ],
        "fault_families": [
            "POOR_COVERAGE",
            "UL_INTERFERENCE",
            "RACH_CONTENTION",
            "DU_PROCESSING_PRESSURE",
            "PRACH_CONFIGURATION",
        ],
    },

    "rrc_establishment": {
        "classification": "KPI_COMPONENT",
        "formula_measurements": [
            "RRC.ConnEstabAtt.Cause",
            "RRC.ConnEstabSucc.Cause",
        ],
        "fault_families": [
            "UPSTREAM_RACH",
            "POOR_COVERAGE",
            "RADIO_INTERFERENCE",
            "RADIO_RESOURCE_CONGESTION",
            "DU_PROCESSING_PRESSURE",
            "CU_CP_PROCESSING_PRESSURE",
            "RRC_CONFIGURATION",
            "SOFTWARE_EVENT",
        ],
    },

    "ng_signalling": {
        "classification": "KPI_COMPONENT",
        "formula_measurements": [
            "UECNTX.ConnEstabAtt.Cause",
            "UECNTX.ConnEstabSucc.Cause",
        ],
        "fault_families": [
            "UPSTREAM_RRC",
            "CU_CP_PROCESSING_PRESSURE",
            "N2_CONGESTION",
            "N2_PACKET_LOSS_LATENCY",
            "SCTP_INSTABILITY",
            "N2_PEER_CONFIGURATION",
            "SOFTWARE_EVENT",
            "AMF_FACING_BOUNDARY",
        ],
    },

    "initial_drb_establishment": {
        "classification": "KPI_COMPONENT",
        "formula_measurements": [
            "DRB.InitialEstabAtt.5QI",
            "DRB.InitialEstabSucc.5QI",
        ],
        "fault_families": [
            "UPSTREAM_RRC_NG",
            "POOR_RADIO_QUALITY",
            "RADIO_RESOURCE_CONGESTION",
            "ADMISSION_RESOURCE_PRESSURE",
            "DU_PROCESSING_PRESSURE",
            "CU_UP_RESOURCE_PRESSURE",
            "QOS_CONFIGURATION",
            "INTERNAL_INTERFACE",
            "SOFTWARE_CONFIGURATION_EVENT",
        ],
    },

    "drb_establishment": {
        "classification": "KPI_COMPONENT",
        "formula_measurements": [
            "DRB.EstabAtt.5QI",
            "DRB.EstabSucc.5QI",
            "DRB.EstabAtt.SNSSAI",
            "DRB.EstabSucc.SNSSAI",
        ],
        "fault_families": [
            "RADIO_RESOURCE_CONGESTION",
            "POOR_RADIO_QUALITY",
            "FIVE_QI_SPECIFIC",
            "SNSSAI_SPECIFIC",
            "DU_PROCESSING_PRESSURE",
            "CU_UP_RESOURCE_PRESSURE",
            "INTERNAL_INTERFACE",
            "SOFTWARE_CONFIGURATION_EVENT",
        ],
    },

    "rrc_resume": {
        "classification": "KPI_COMPONENT",
        "formula_measurements": [
            "RRC.ResumeAtt.Cause",
            "RRC.ResumeSucc.Cause",
            "RRC.ResumeSuccByFallback.cause",
            "RRC.ResumeFallbackToSetupAtt.cause",
            "RRC.ReEstabSuccWithoutUeContext",
            "RRC.ReEstabFallbackToSetupAtt",
        ],
        "fault_families": [
            "RADIO_RESUME_DEGRADATION",
            "RACH_RESUME_ACCESS",
            "CU_CP_PROCESSING_PRESSURE",
            "UE_CONTEXT_UNAVAILABLE",
            "FALLBACK_ABNORMALITY",
            "REESTABLISHMENT_ABNORMALITY",
            "RESUME_CONFIGURATION",
            "CU_RESTART_SOFTWARE_EVENT",
        ],
    },

    "drb_resume": {
        "classification": "KPI_COMPONENT",
        "formula_measurements": [
            "DRB.ResumeAtt.5QI",
            "DRB.ResumeSucc.5QI",
        ],
        "fault_families": [
            "UPSTREAM_RRC_RESUME",
            "POOR_RADIO_QUALITY",
            "RADIO_RESOURCE_CONGESTION",
            "BEARER_CONTEXT_UNAVAILABLE",
            "DU_PROCESSING_PRESSURE",
            "CU_UP_RESOURCE_PRESSURE",
            "FIVE_QI_SPECIFIC",
            "INTERNAL_INTERFACE",
            "SOFTWARE_CONFIGURATION_EVENT",
        ],
    },
}

print(
    "Reference components:",
    len(ACCESSIBILITY_OBSERVABILITY_COMPONENTS_V1)
)
```


#### Closing Design Principle

The synthetic environment must never reduce troubleshooting to:

```text
KPI degraded
    ↓
pre-programmed root cause
```

It must instead produce:

```text
KPI degraded
    ↓
multiple plausible hypotheses
    ↓
standards formula evidence
    +
diagnostic network evidence
    +
topology / chronology context
    ↓
agent investigation
    ↓
evidence-supported diagnosis
```

> **The LLM orchestrates the investigation. The environment must supply enough network truth for the diagnosis to be discovered rather than guessed.**
